In [ ]:
# Uninstall to ensure a clean slate and resolve dependencies
!pip uninstall -y transformers datasets tokenizers accelerate sentence-transformers huggingface-hub

# Reinstall latest compatible versions
!pip install --no-cache-dir transformers
!pip install --no-cache-dir datasets
!pip install --no-cache-dir accelerate
!pip install --no-cache-dir evaluate seqeval

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)
from evaluate import load

In [ ]:
dataset = load_dataset("wikiann", "en")

print(dataset["train"][0])
print(dataset["train"].features)

In [ ]:
label_column = "ner_tags"
label_list = dataset["train"].features[label_column].feature.names

print("Labels:", label_list)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,

)

In [ ]:
trainer.train()

In [ ]:
metric = load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }


In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()
print("Evaluation Results:", results)

In [ ]:
sentence = "John works at Google in California"

inputs = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs).logits

predictions = np.argmax(outputs.detach().numpy(), axis=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
predicted_tags = [label_list[p] for p in predictions[0]]

print("\nInference:")
for token, tag in zip(tokens, predicted_tags):
    print(f"{token:15} -> {tag}")